# Model Training for Classification with HydraBLE (Variable Length Finetuning)

In [1]:
import os
from pathlib import Path


os.chdir(Path.cwd().parents[0])
print("Now in:", Path.cwd())

dataPathProcessed = str(Path.cwd()) + r"\data\csv" + r"\Processed Files\\"
checkPointPath = str(Path.cwd()) + r"\out\modeling\checkpoints\classification_finetuning\\"

Now in: C:\Users\stsax\OneDrive\Studium\9. Semester\Masterarbeit\Repository


In [2]:
MAX_TOKEN_LENGTH = 1024
MAX_SEQUENCE_LENGTH = 32

In [3]:
import pickle
import pandas as pd
from modeling.BleDataset import BLEStreamDataset
import torch

from bpe.bpe import BleBytePairEncoder
from modeling.Trainer import TrainerConfig

MaskConfigPath = str(Path.cwd()) + r"\data_masking\mask_configs\\"
picklePath = str(Path.cwd()) + r"\out\pickle_objects\bpe" + "\\"

with open(picklePath + 'Fitted_BPE_State_Dict.pickle', 'rb') as f:
    state_dict = pickle.load(f)


pkt_df_train = pd.read_parquet(dataPathProcessed + r"classification_dataset_v2_train.parquet")
seq_table_train = pd.read_parquet(dataPathProcessed + r"classification_sequence_table_v2_train.parquet")
stream_idx_train = pd.read_parquet(dataPathProcessed + r"classification_stream_index_v2_train.parquet")


pkt_df_val = pd.read_parquet(dataPathProcessed + r"classification_dataset_v2_validation.parquet")
seq_table_val = pd.read_parquet(dataPathProcessed + r"classification_sequence_table_v2_validation.parquet")
stream_idx_val = pd.read_parquet(dataPathProcessed + r"classification_stream_index_v2_validation.parquet")



dataset_train = BLEStreamDataset(packet_df=pkt_df_train, sequence_table=seq_table_train, stream_index=stream_idx_train,
                           max_sequence_length=MAX_SEQUENCE_LENGTH, max_token_length=MAX_TOKEN_LENGTH,
                           tokenizer_dict=state_dict, mask_config_path=MaskConfigPath, dataset_type='train', augment_data=True, adapt_sequence_length=True, deterministic=False)

dataset_val = BLEStreamDataset(packet_df=pkt_df_val, sequence_table=seq_table_val, stream_index=stream_idx_val,
                           max_sequence_length=MAX_SEQUENCE_LENGTH, max_token_length=MAX_TOKEN_LENGTH,
                           tokenizer_dict=state_dict, mask_config_path=MaskConfigPath, dataset_type='validation', augment_data=True, adapt_sequence_length=True)

In [4]:
train_loader = torch.utils.data.DataLoader(dataset_train, batch_size=50, num_workers=4, pin_memory=True,     persistent_workers=True,     drop_last=False,     prefetch_factor=4, shuffle=True)

validation_loader = torch.utils.data.DataLoader(dataset_val, batch_size=50, num_workers=4, pin_memory=True,     persistent_workers=True,     drop_last=False,     prefetch_factor=4, shuffle=False)

In [5]:
from modeling.HydraBLE import HydraBLETransformer

bpe = BleBytePairEncoder.from_state_dict(state_dict)

model = HydraBLETransformer(
        vocab_size=bpe.vocab_size,
        num_classes=dataset_train.num_of_known_classes,
        pad_id=1,
        max_heads=8,
        head_dim=64,
        depth=4,
        max_len=2048,
        subnet_heads=(1, 2, 4, 8),
        separate_cls_heads=True,
    )


ckpt = torch.load(str(Path.cwd()) + r"\out\modeling\checkpoints\classification_data_augmented\\"  + "last.pt", map_location="cuda:0")
model.load_state_dict(ckpt["model_state_dict"])



<All keys matched successfully>

In [6]:
from modeling.Trainer import ClassificationTrainer

config = TrainerConfig()

config.checkpoint_dir = checkPointPath
config.lr = 1e-2 * 0.5
config.epochs = 3
config.device =  "cuda:0"

trainer = ClassificationTrainer(model, train_loader, validation_loader, config)

In [7]:
trainer.fit()

49it [01:17,  1.53s/it]

[train] epoch=1 step=50 active_heads=1 loss=1.7329


99it [02:33,  1.60s/it]

[train] epoch=1 step=100 active_heads=4 loss=1.6417


149it [03:53,  1.23s/it]

...

[train] epoch=3 step=550 active_heads=8 loss=1.4385


599it [16:06,  1.73s/it]

[train] epoch=3 step=600 active_heads=2 loss=1.4391


600it [16:06,  1.61s/it]



[epoch 3] train_loss=1.4391 
  [val h=8] loss=1.4110 
  best_loss@h=8: 1.4110

